# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** *A page is worth refreshing if it has real traffic worth
saving (visible) and it's old enough that staleness is a plausible cause (stale) — and if it's
already fading within the current month, that's the strongest version of the same story.*

This leans on two signals, checked below before they're trusted:

1. **Staleness** — the signal behind FlyRank's refresh flags. Older content should be more
   likely to be declining.
2. **Volume** — the signal behind FlyRank's quick-win logic. High-impression pages are worth
   prioritizing over low-traffic ones with the same trend, because the upside from fixing them
   is bigger.

**Real verdicts, from the live warehouse (Feb 2026 panel, n = 145,279):**

- **Staleness — MIXED.** Decline rate by age: `<90d` 24.4%, `90-180d` 27.8%, `180-365d` 27.8%,
  `365-730d` 17.9% (nothing older exists yet in this release — earliest `content_created_date`
  is Oct 2024, so no row has crossed 730 days as of Feb 2026). Not monotonic — the oldest
  surviving band actually declines *least*, not most. The real risk band is the *middle* ages
  (90-365 days), not "older is worse." The rule below uses a 90-365 day band, not a `>= 365`
  floor, because that's what this table actually supports.
- **Volume — OPPOSITE (as a decline predictor).** Decline rate falls cleanly as volume rises:
  `<100` 34.9% -> `100-500` 21.0% -> `500-2000` 17.1% -> `2000-10000` 15.2% -> `10000+` 16.0%.
  Bigger pages are less likely to hit this decline definition, almost certainly because
  small-sample swings are noisier at low volume. This doesn't support "more traffic predicts
  more decline" -- but it does justify using a volume floor as a noise gate, not as a priority
  signal in its own right. The rule keeps `feb_impressions` as a size-of-opportunity multiplier
  among already-flagged pages, not as evidence a page is more likely to be a true positive.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
from getpass import getpass

# --- Auth: register the HF read token as a DuckDB secret. Never paste a token into a cell ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"

FEATURE_MONTH = "2026-02"  # same mid-panel month as the data contract
LABEL_MONTH = "2026-03"    # same forward month as the data contract

# --- Rebuild the exact w03 panel: Feb features -> March forward outcome ---
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions,
           SUM(CASE WHEN report_date < DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()
panel["declined_next_month"] = (
    panel["mar_impressions"] < panel["feb_impressions"] * 0.8
).astype(int)
print(f"Panel rows (matches w03 exactly): {len(panel):,}")

# --- Join content age for the staleness signal ---
# Real schema confirmed via the error message on first run: dim_content uses
# "content_created_date" (not "content_created_at") -- also available: content_updated_date.
content_meta = con.sql(f"""
    SELECT content_hash_id, content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

panel = panel.merge(content_meta, on="content_hash_id", how="left")
panel["content_created_date"] = pd.to_datetime(panel["content_created_date"])
as_of = pd.Timestamp(f"{FEATURE_MONTH}-28")
panel["age_days"] = (as_of - panel["content_created_date"]).dt.days
print(f"Rows with resolvable age: {panel['age_days'].notna().sum():,} / {len(panel):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel rows (matches w03 exactly): 145,279
Rows with resolvable age: 145,279 / 145,279


In [7]:
print("== Signal check A: staleness (age_days) -- behind FlyRank's refresh flags ==")
age_bins = [-1, 90, 180, 365, 730, 100000]
age_labels = ["<90d", "90-180d", "180-365d", "365-730d", "730d+"]
panel["age_tier"] = pd.cut(panel["age_days"], bins=age_bins, labels=age_labels)

staleness_table = panel.groupby("age_tier", observed=True).agg(
    n=("declined_next_month", "size"),
    decline_rate=("declined_next_month", "mean"),
).reset_index()
print(staleness_table.to_string(index=False))
print("\nVerdict (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE")

print("\n== Signal check B: volume (feb_impressions) -- behind FlyRank's quick-win logic ==")
vol_bins = [-1, 100, 500, 2000, 10000, 10_000_000]
vol_labels = ["<100", "100-500", "500-2000", "2000-10000", "10000+"]
panel["volume_tier"] = pd.cut(panel["feb_impressions"], bins=vol_bins, labels=vol_labels)

volume_table = panel.groupby("volume_tier", observed=True).agg(
    n=("declined_next_month", "size"),
    decline_rate=("declined_next_month", "mean"),
).reset_index()
print(volume_table.to_string(index=False))
print("\nVerdict (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE")

== Signal check A: staleness (age_days) -- behind FlyRank's refresh flags ==
age_tier     n  decline_rate
    <90d 43372      0.244213
 90-180d 27814      0.277918
180-365d 63364      0.278123
365-730d 10729      0.179047

Verdict (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE

== Signal check B: volume (feb_impressions) -- behind FlyRank's quick-win logic ==
volume_tier     n  decline_rate
       <100 68623      0.348513
    100-500 30514      0.210395
   500-2000 25250      0.171327
 2000-10000 17578      0.152122
     10000+  3314      0.159928

Verdict (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**One rule, one reason code path, one action label** — no fitted weights, just readable
conditions multiplied together (mirrors the `building-baselines` skill's own example):

`baseline_score = stale * visible * feb_impressions`

`stale` uses a 365-day threshold and `visible` uses a 500-impression threshold — provisional
defaults, to be adjusted against the bucket tables above once real verdicts come back. A third,
already-safe signal (declining within February itself, `feb_h2` vs `feb_h1` — built the same
way as the honest check in the w03 leakage trap, so it uses no March data at all) splits the
reason code into a stronger and weaker version of the same story, without changing the ranking
score itself.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import json
from pathlib import Path

# Thresholds set from the real bucket tables in Section 1, not guessed:
# staleness was MIXED with a non-monotonic pattern -- the actual elevated-risk band is
# 90-365 days (90-180d and 180-365d both ~27.8% decline, vs 24.4% under 90d and 17.9% over
# 365d) -- so "stale" is a band, not a ">= 365 days" floor. Volume was OPPOSITE as a decline
# predictor, but the >=500 gate is still justified as a noise filter: <100 impressions is 34.9%
# decline (mostly small-sample noise), 500+ settles into a stable ~15-17% range.
STALE_MIN_DAYS = 90
STALE_MAX_DAYS = 365
VISIBLE_THRESHOLD_IMPR = 500

panel["stale"] = (
    (panel["age_days"] >= STALE_MIN_DAYS) & (panel["age_days"] < STALE_MAX_DAYS)
).astype(int)
panel["visible"] = (panel["feb_impressions"] >= VISIBLE_THRESHOLD_IMPR).astype(int)
panel["declining_now"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)

# The rule: readable, multiplicative, no fitted weights.
panel["baseline_score"] = panel["stale"] * panel["visible"] * panel["feb_impressions"]


def reason_code(row):
    if row["stale"] and row["visible"] and row["declining_now"]:
        return "stale_visible_declining"
    if row["stale"] and row["visible"]:
        return "stale_visible_page"
    return "not_flagged"


def suggested_action(reason):
    return "refresh" if reason != "not_flagged" else "monitor"


panel["reason_code"] = panel.apply(reason_code, axis=1)
panel["suggested_action"] = panel["reason_code"].apply(suggested_action)
panel["baseline_rank"] = (
    panel["baseline_score"].rank(method="first", ascending=False).astype(int)
)

n_flagged = (panel["reason_code"] != "not_flagged").sum()
print(f"Pages flagged for refresh: {n_flagged:,} / {len(panel):,} ({n_flagged/len(panel):.1%})")
print(panel["reason_code"].value_counts().to_string())

out_cols = [
    "client_hash_id", "content_hash_id", "feb_impressions", "age_days",
    "stale", "visible", "declining_now", "baseline_score",
    "reason_code", "suggested_action", "baseline_rank",
]
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
panel.sort_values("baseline_rank")[out_cols].to_csv(out_path, index=False)
print(f"\nWrote {out_path} ({len(panel):,} rows)")

Pages flagged for refresh: 31,131 / 145,279 (21.4%)
reason_code
not_flagged                114148
stale_visible_page          26768
stale_visible_declining      4363

Wrote work/outputs/baseline_action_score.csv (145,279 rows)


In [9]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


K = 20
p_at_k = precision_at_k(panel["baseline_score"].values, panel["declined_next_month"].values, K)
base_rate = panel["declined_next_month"].mean()

print(f"Precision@{K}: {p_at_k:.1%}")
print(f"Base rate (declined_next_month.mean()): {base_rate:.1%}")
print(f"Lift over base rate: {p_at_k / base_rate:.2f}x" if base_rate > 0 else "Base rate is 0")

metrics = {
    "feature_month": FEATURE_MONTH,
    "label_month": LABEL_MONTH,
    "k": K,
    "precision_at_k": float(p_at_k),
    "base_rate": float(base_rate),
    "n_total": int(len(panel)),
    "n_flagged": int(n_flagged),
    "stale_min_days": STALE_MIN_DAYS,
    "stale_max_days": STALE_MAX_DAYS,
    "visible_threshold_impressions": VISIBLE_THRESHOLD_IMPR,
    "signal_checks": {
        "staleness": staleness_table.assign(
            n=lambda d: d["n"].astype(int),
            decline_rate=lambda d: d["decline_rate"].astype(float),
        ).to_dict(orient="records"),
        "volume": volume_table.assign(
            n=lambda d: d["n"].astype(int),
            decline_rate=lambda d: d["decline_rate"].astype(float),
        ).to_dict(orient="records"),
    },
}
metrics_path = Path("work/outputs/baseline_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nWrote {metrics_path} -- this one IS worth committing (it's the run's receipt).")

Precision@20: 30.0%
Base rate (declined_next_month.mean()): 26.1%
Lift over base rate: 1.15x

Wrote work/outputs/baseline_metrics.json -- this one IS worth committing (it's the run's receipt).


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = panel.sort_values("baseline_rank").head(20)[
    ["baseline_rank", "content_hash_id", "feb_impressions", "age_days",
     "reason_code", "suggested_action", "declined_next_month"]
]
top20

,baseline_rank,content_hash_id,feb_impressions,age_days,reason_code,suggested_action,declined_next_month
17322,1,content_9c057b66c30a3abb,195648.0,212,stale_visible_page,refresh,1
18295,2,content_512dbad65bd5ade9,167303.0,156,stale_visible_page,refresh,0
110050,3,content_e8a52cf3d5988c07,162129.0,199,stale_visible_page,refresh,0
24003,4,content_e7b5dd4dff461ad2,154502.0,312,stale_visible_page,refresh,0
31442,5,content_c9a0c2fdbdbfb562,142215.0,344,stale_visible_declining,refresh,1
30292,6,content_29c4a3831609805d,129662.0,229,stale_visible_page,refresh,1
18445,7,content_db122b8ba22641b8,127941.0,156,stale_visible_page,refresh,1
14644,8,content_471d9cabce329a66,119240.0,344,stale_visible_page,refresh,0
117972,9,content_b556f0bd87d6fcca,116206.0,212,stale_visible_declining,refresh,0
17229,10,content_4d0d79fc12632ef8,113798.0,212,stale_visible_declining,refresh,1


**The real pattern this exposes:** because `baseline_score = stale x visible x feb_impressions`,
ranking is driven almost entirely by raw page size. 14 of the 20 are `stale_visible_page` (no
Feb-internal decline evidence at all) and still outrank smaller `stale_visible_declining` pages
purely because they're bigger. Only 4 of the 6 true hits (ranks 5, 10 — and arguably the model
should be finding more of them higher) actually carry the `declining_now` evidence the rule was
supposed to reward; ranks 6, 7, 16 hit the label by chance, not by internal evidence.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("== Leakage check ==")
score_inputs = {"age_days", "feb_impressions", "stale", "visible", "declining_now"}
label_or_future_cols = {"declined_next_month", "mar_impressions", "feb_h1", "feb_h2"}
# feb_h1/feb_h2 build declining_now (a derived signal, itself Feb-only) but are never inputs
# to baseline_score directly -- only stale * visible * feb_impressions are.
leak = score_inputs & {"declined_next_month", "mar_impressions"}
print("Score inputs that touch the label or a future window (should be empty):", leak or "none")
print("declining_now is built only from feb_h1/feb_h2 (both February-only, no March) --",
      "safe by construction, same pattern as the w03 leakage trap's honest check.")

print("\n== FlyRank product-flag check ==")
product_flag_cols = {"health_score", "priority_score", "action_type", "refresh_tier"}
used_cols = set(panel.columns)
print("Product-decision columns present in this panel (should be empty):",
      product_flag_cols & used_cols or "none")

== Leakage check ==
Score inputs that touch the label or a future window (should be empty): none
declining_now is built only from feb_h1/feb_h2 (both February-only, no March) -- safe by construction, same pattern as the w03 leakage trap's honest check.

== FlyRank product-flag check ==
Product-decision columns present in this panel (should be empty): none


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.